# 4.2.2 Clustering Stromal fraction

**Inés Sentís**

Date of execution

In [ ]:
Sys.Date()

## Libraries

In [ ]:
suppressPackageStartupMessages({
library(Seurat)
library(Signac)
library(tidyverse)
library(harmony)
library(scater) 
library(scran)
library(dittoSeq)
library(here)
library(glue)})

## Parameters

In [ ]:
#here::dr_here(show_reason = TRUE)
source(here("utils/bin.R"))
source(here("utils/paths.R"))
source(here("AVGUSTINOVA_02_to_30/utils/style.R"))
source(here("AVGUSTINOVA_02_to_30/utils/variables.R"))

"{int}/{robj_dir}" %>%
    glue::glue() %>%
    here::here() %>%
    dir.create(path = .,
               showWarnings = FALSE,
               recursive = TRUE)
set.seed(0)

## Load Data

In [ ]:
combined_obj <- readRDS(here(glue("{int}/{robj_dir}/4.0.5-integrated_filt.rds")))

In [ ]:
unique(combined_obj$cell_groups)

In [ ]:
DefaultAssay(combined_obj) <- "RNA"

In [ ]:
combined_obj <- subset(combined_obj, subset=cell_groups=="Other stromal")

combined_obj <- DietSeurat(combined_obj,
                           counts = TRUE,
                           data = TRUE,
                           assays = c("RNA"),
                           dimreducs = NULL,
                           graphs = NULL)

## Remove features with no counts

In [ ]:
keep_genes <- data.frame(rowSums(as.matrix(combined_obj[['RNA']]@counts)) >= 1)
colnames(keep_genes) <- "logical"
keep_genes <- subset(keep_genes, logical==TRUE)
combined_obj[['RNA']]@counts <- combined_obj[['RNA']]@counts[rownames(keep_genes), ]
combined_obj[['RNA']]@data <- combined_obj[['RNA']]@data[rownames(keep_genes), ]

In [ ]:
combined_obj <- combined_obj %>% NormalizeData(
                normalization.method = "LogNormalize",
                scale.factor = 1e4) 

### High variable features

Remove genes that might influence biological variability

In [ ]:
genes_dmf <- c("XACT", "HTR2C",
               "XIST", "DDX3Y","USP9Y","NLGN4Y", "KDM5D", "EIF1AY", "UBA1")
mt_genes <- grep("^MT-", rownames(combined_obj[["RNA"]]), value = TRUE)
genes_out <- c(genes_dmf, top_c, mt_genes) 

In [ ]:
list_objects <- SplitObject(combined_obj, split.by = "patient")

In [ ]:
find_hvgs <-  function(obj) {
    
    pat <- unique(obj$patient)
    
    # remove genes from matrix before modulating variance and mean
    DefaultAssay(obj) <- "RNA"
    df_meta <- obj@meta.data
    counts <- obj[['RNA']]@counts
    counts <- counts[-(which(rownames(counts) %in% genes_out)),]
    obj2 <- CreateSeuratObject(counts = counts, meta.data = df_meta)
    
    # normalize again
    obj2 <- obj2 %>% 
            NormalizeData(
                normalization.method = "LogNormalize",
                scale.factor = 1e4) 
    
    # compute the variance and mean of the log-expression values 
    sce <- as.SingleCellExperiment(obj2)
    gene_var <- modelGeneVar(sce)
    
    tops <- gene_var %>% 
        as.data.frame() %>% 
        arrange(desc(total)) %>% 
        head(n=20)

    plt <- gene_var %>% 
      as.data.frame() %>% 
      ggplot(aes(mean, total)) +
      geom_point() +
      geom_line(aes(y = tech), colour = "dodgerblue", size = 1) +
      labs(x = "Mean of log-expression", y = "Variance of log-expression")+
      geom_text(data=tops, aes(mean,total,label=rownames(tops))) + 
      ggtitle(pat) + 
      theme(plot.title = element_text(hjust = 0.5, size = 12, face = "bold"))

    hvgs <- getTopHVGs(gene_var,fdr.threshold = 0.05)
    return(list(plot = plt, hvgs_pat = hvgs))
}

In [ ]:
modelling <- lapply(list_objects, find_hvgs)

In [ ]:
list_plots <- lapply(modelling, function(el) {
    el$plot
})

In [ ]:
options(repr.plot.width = 20, repr.plot.height = 15)
cowplot::plot_grid(plotlist = list_plots,
                   align = "hv",
                   axis = "trbl",
                   ncol=4)

In [ ]:
list_hvg <- lapply(modelling, function(el) {
    hvg <- el$hvgs_pat
    print(length(hvg))
    hvg
})

In [ ]:
hvg_common <- find_perc(list_hvg, perc = 2/11)
length(hvg_common)

### Normalization and linear dimensional reduction

In [ ]:
VariableFeatures(combined_obj) <- hvg_common
 
combined_obj <- combined_obj %>% 
  NormalizeData(normalization.method = "LogNormalize",
                scale.factor = 1e4) %>%
  ScaleData(features=hvg_common) %>% 
  RunPCA(features=hvg_common) 

In [ ]:
options(repr.plot.width = 10, repr.plot.height = 8)
ElbowPlot(combined_obj, ndims=50)

In [ ]:
rna_comp = 1:25

In [ ]:
options(repr.plot.width = 20, repr.plot.height = 6)
VizDimLoadings(combined_obj, dims = 1:5, reduction = "pca", ncol=5)

In [ ]:
options(repr.plot.width = 20, repr.plot.height = 10)

DimPlot(combined_obj, reduction = "pca", group.by = c("clusters_donor","patient"), order=T, label=T)

In [ ]:
combined_obj <- RunUMAP(
  combined_obj,
  dims = rna_comp,
  reduction = "pca",
  reduction.name = "umap",
  reduction.key = "UMAP_"
)

In [ ]:
options(repr.plot.width = 20, repr.plot.height = 6)
p1 <- DimPlot(
  combined_obj,
   reduction = "umap",
   group.by = "patient",
   pt.size = 0.1,
   cols = colors_patients,
) + ggtitle('RNA UMAP\nby patient (no integration)') + 
    theme(plot.title = element_text(hjust = 0.5, size = 12, face = "bold")) 

p2 <- DimPlot(
  combined_obj,
   reduction = "umap",
   group.by = "subproj",
   pt.size = 0.1
) + ggtitle('RNA UMAP\nby subproject (no integration)') + 
    theme(plot.title = element_text(hjust = 0.5, size = 12, face = "bold"))

p3 <- DimPlot(
  combined_obj,
   reduction = "umap",
   group.by = "clusters_donor",
   pt.size = 0.1
) + ggtitle('RNA UMAP\nby level 1 (no integration)') + 
    theme(plot.title = element_text(hjust = 0.5, size = 12, face = "bold"))

patchwork::wrap_plots(list(p1,p2,p3), ncol=3)

## Integration with Harmony

In [ ]:
# make a copy of the PCA reduction
combined_obj@reductions$pca_filt <- combined_obj@reductions$pca

# Filter to the desired number of components according to elbow
combined_obj@reductions$pca_filt@cell.embeddings <- combined_obj@reductions$pca_filt@cell.embeddings[, rna_comp]

combined_obj@reductions$pca_filt@feature.loadings <- combined_obj@reductions$pca_filt@feature.loadings[, rna_comp]


combined_obj <- RunHarmony(
  object = combined_obj,
  reduction = "pca_filt",
  dims = rna_comp,
  group.by.vars = c("patient"),
  assay.use = "RNA",
  max_iter = 15,
  project.dim = T,
  reduction.save = "harmony_pca"
)

In [ ]:
combined_obj <- RunUMAP(
  combined_obj,
  dims = rna_comp,
  reduction = "harmony_pca",
  reduction.name = "umap.rna.harmony",
  reduction.key = "RNAhar_",
  assay = "RNA")

In [ ]:
options(repr.plot.width = 9, repr.plot.height = 8)
DimPlot(
   combined_obj,
   reduction = "umap.rna.harmony",
   group.by = "patient",
   pt.size = 0.05,
   cols= colors_patients,
) + ggtitle('RNA UMAP\nby patient') + 
    theme(plot.title = element_text(hjust = 0.5, size = 12, face = "bold")) 

In [ ]:
options(repr.plot.width = 20, repr.plot.height = 10)
FeaturePlot(combined_obj, reduction = "umap.rna.harmony",
                features = c("COL1A1","PECAM1","PODXL", "CAV1", "LAMA2", "PDGFRB"), 
                ncol=4, min.cutoff = "q05", max.cutoff = "q95")

## Clustering

In [ ]:
# Find the K-nearest neighbors
combined_obj <- FindNeighbors(combined_obj, reduction = "harmony_pca", 
                            dims = rna_comp)

In [ ]:
combined_obj <- FindClusters(combined_obj, resolution = 0.3, graph.name = "RNA_snn")

In [ ]:
options(repr.plot.width = 9, repr.plot.height = 8)
DimPlot(combined_obj, reduction = "umap.rna.harmony", label=T) +
    labs(title="Clustering with Harmony") + 
    theme(plot.title = element_text(hjust = 0.3, size = 12, face = "bold"))

In [ ]:
rna.markers <- FindAllMarkers(combined_obj, max.cells.per.ident=1000,
                              logfc.threshold = 0.1, only.pos = TRUE)

In [ ]:
rna.markers <- rna.markers %>% 
  dplyr::filter(p_val_adj < 0.05) %>% 
  arrange(desc(avg_log2FC))

Tumoral mixed markers...

In [ ]:
c="5"
rna.markers %>% 
    filter(cluster == c) %>% 
    head(n=30)

In [ ]:
rel <- dittoBarPlot(combined_obj, "clusters_donor", group.by = "ident", 
                      retain.factor.levels = TRUE) + 
            labs(title = "Relative proportion of \n clusters cells per patient") + 
            theme(plot.title = element_text(hjust = 0.5, size = 16, face = "bold"),
            text =  element_text(hjust = 0.5,size = 14))

abs <- dittoBarPlot(combined_obj, "clusters_donor", group.by = "ident", 
            scale = "count", retain.factor.levels = TRUE) + 
            labs(title ="Absolute proportion of \n clusters cells per patient") + 
            theme(plot.title = element_text(hjust = 0.5, size = 16, face = "bold"),
            text =  element_text(hjust = 0.5, size = 14))+ NoLegend()

In [ ]:
options(repr.plot.width = 12, repr.plot.height = 6, warn=-1,verbose = FALSE)
abs+rel

In [ ]:
options(repr.plot.width = 15, repr.plot.height = 5, warn=-1,verbose = FALSE)
VlnPlot(combined_obj, feature=c("nCount_RNA", "nFeature_RNA","doublet_score"),ncol=3, pt.size = 0.01)

In [ ]:
options(repr.plot.width = 20, repr.plot.height = 5, warn=-1,verbose = FALSE)
VlnPlot(combined_obj, feature=c("ACTA2", "TAGLN","RGS5","MYH11" ),ncol=4, pt.size = 0.01)

In [ ]:
options(repr.plot.width = 9, repr.plot.height = 8)
DimPlot(
   combined_obj,
   reduction = "umap.rna.harmony",
   group.by = "clusters_donor",
   pt.size = 0.05,
) + ggtitle('RNA UMAP\nby patient') + 
    theme(plot.title = element_text(hjust = 0.5, size = 12, face = "bold")) 

## Low quality cells

In [ ]:
combined_obj <- subset(combined_obj, idents=c("4"), invert=T)

combined_obj <- DietSeurat(combined_obj,
                           counts = TRUE,
                           data = TRUE,
                           assays = c("RNA"),
                           dimreducs = NULL,
                           graphs = NULL)

In [ ]:
keep_genes <- data.frame(rowSums(as.matrix(combined_obj[['RNA']]@counts)) >= 1)
colnames(keep_genes) <- "logical"
keep_genes <- subset(keep_genes, logical==TRUE)
combined_obj[['RNA']]@counts <- combined_obj[['RNA']]@counts[rownames(keep_genes), ]
combined_obj[['RNA']]@data <- combined_obj[['RNA']]@data[rownames(keep_genes), ]

## Save

In [ ]:
saveRDS(combined_obj, here(glue("{int}/{robj_dir}/4.2.2-stromal_fraction.rds")))

In [ ]:
sessionInfo()